# IRIS-X：完整離線 G4 ARC 小模型試驗

此版本只讀三個已固定版本的 private Kaggle Datasets：Granite 權重、CUDA 12.8／Python 3.12 wheels、獨立版本的程式與 32+4 個 demonstration-only task。所有開關預設 False。安裝、驗證、訓練與推論都沒有網路下載步驟。

依你的明確指示，保留既有 ARC-AGI-3 比賽資料掛載。此 ARC-AGI-2 程式不讀取或使用該比賽資料；保留掛載沒有啟動 GPU、訓練或付費執行。

Granite 官方索引與 shard header 不一致；此材料保留官方原始索引與來源紀錄，只重建 shard 對應及位元組總數。權重、config、tokenizer 和完整詞彙表都不變。完整模型載入與 GPU 訓練仍未驗證。

訓練目標仍為一張 G4 RTX PRO 6000 Blackwell。Kaggle 此處只保存來源，GPU OFF、Internet OFF，沒有建立 VM、付费執行或正式提交。首次 GPU 訓練與四張 L4 部署仍須手動驗證。

手動順序：先 VERIFY_OFFLINE_INPUTS；再 INSTALL_OFFLINE_ENVIRONMENT；再 CHECK_CPU_IMPORTS；最後才依需求單獨選 smoke2 訓練、A/B/C、獨立評分或匯出。安裝到獨立 venv，無須修改 Kaggle 既有套件或重啟 kernel。目標必須 Linux x86_64、CPython 3.12、glibc ≥2.28。準備機下載與 Dataset 上傳已在此 notebook 之外完成。

在 G4 使用時，先把相同三個 Dataset 完整複製到當地；填入三個明確資料夾路徑。缺檔、hash／版本不同、Python ABI 不合即停止，不會連網補檔。停掉程序不代表停止 VM 計費。


In [ ]:
from pathlib import Path
VERIFY_OFFLINE_INPUTS = False
INSTALL_OFFLINE_ENVIRONMENT = False
CHECK_CPU_IMPORTS = False
RUN_CALIBRATION = False
ENABLE_TRAINING_PILOT = False
RUN_ARCHITECTURE_PILOT = False
SCORE_DEVELOPMENT = False
EXPORT_REPORT = False
RUNTIME_ROOT = None  # 非 Kaggle 主機填入完整資料夾；不要填下載 URL
MODEL_ROOT = None
WHEELS_ROOT = None
TRAINING_PROFILE = 'smoke2'
RUN_TAG = 'granite-offline-smoke2-001'
MAX_WALL_SECONDS = 1800
APPROVED_MAX_COST_USD = None
ESTIMATED_HOURLY_COST_USD = None
ADAPTER_PATH = None
BASE = (Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()) / 'iris-next-offline'
ENVIRONMENT = BASE / 'environment-cp312-cu128'


In [ ]:
PINS = {'model': {'manifest_sha256': 'cd845822cd2c0ca749dd3e6f580f51e68b91c0e5c5886fe239fc3c14f5bb0b99', 'private': True, 'reference': 'danielwuu/iris-arc-granite42-8b-offline/2'}, 'runtime': {'manifest_sha256': 'ab4e3058f56df5878627520adcf1d172f4121bfed8309c3de0afcefb482f391b', 'private': True, 'reference': 'danielwuu/iris-arc-next-runtime/4'}, 'wheels': {'manifest_sha256': '2b082a3c36534b5602b8a9f738beec2c01b3d2df11b5438f1c352a84c118b052', 'private': True, 'reference': 'danielwuu/iris-arc-next-cu128-cp312-wheels/1'}}
FLAGS = ('VERIFY_OFFLINE_INPUTS', 'INSTALL_OFFLINE_ENVIRONMENT', 'CHECK_CPU_IMPORTS', 'RUN_CALIBRATION', 'ENABLE_TRAINING_PILOT', 'RUN_ARCHITECTURE_PILOT', 'SCORE_DEVELOPMENT', 'EXPORT_REPORT')
import hashlib, json, re, sys
if any(type(globals()[f]) is not bool for f in FLAGS): raise ValueError('所有手動開關必須是 bool')
PHASES = ('VERIFY_OFFLINE_INPUTS','INSTALL_OFFLINE_ENVIRONMENT','CHECK_CPU_IMPORTS','RUN_CALIBRATION','RUN_ARCHITECTURE_PILOT','SCORE_DEVELOPMENT','EXPORT_REPORT')
if sum(globals()[f] for f in PHASES) > 1: raise ValueError('每次只開一個 phase')
ACTIVE = any(globals()[f] for f in PHASES)
if ACTIVE:
    sys.dont_write_bytecode = True
    if not re.fullmatch(r'[a-z0-9][a-z0-9-]{2,79}', RUN_TAG): raise ValueError('使用新的安全 RUN_TAG')
    def _root(role, explicit):
        owner, slug, version = PINS[role]['reference'].split('/')
        choices = [Path(explicit)] if explicit is not None else [Path('/kaggle/input/datasets') / owner / slug, Path('/kaggle/input') / slug]
        found = [p for p in choices if p.is_dir()]
        if len(found) != 1: raise ValueError('缺少或重複 Dataset mount: ' + role)
        return found[0]
    RUNTIME_ROOT = _root('runtime', RUNTIME_ROOT)
    MODEL_ROOT = _root('model', MODEL_ROOT)
    WHEELS_ROOT = _root('wheels', WHEELS_ROOT)
    _manifest = RUNTIME_ROOT / 'bundle-manifest.json'
    if hashlib.sha256(_manifest.read_bytes()).hexdigest() != PINS['runtime']['manifest_sha256']: raise ValueError('程式 Dataset 版本/hash 不符')
    _files = json.loads(_manifest.read_bytes())['files']
    for _name, _entry in _files.items():
        _p = RUNTIME_ROOT / _name
        if Path(_name).is_absolute() or '..' in Path(_name).parts or not _p.resolve().is_relative_to(RUNTIME_ROOT.resolve()) or _p.is_symlink() or any(q.is_symlink() for q in _p.parents): raise ValueError('非法 Dataset 路徑')
        if _p.stat().st_size != _entry['bytes'] or hashlib.sha256(_p.read_bytes()).hexdigest() != _entry['sha256']: raise ValueError('程式 Dataset 內容不同: ' + _name)
    _code = RUNTIME_ROOT / 'runtime'
    if 'tools' in sys.modules and any(Path(p).resolve() != (_code / 'tools').resolve() for p in getattr(sys.modules['tools'], '__path__', [])): raise RuntimeError('已有不同 tools package；請用乾淨 kernel 避免來源混用')
    sys.path.insert(0, str(_code))
    from tools.arc_next_offline_bundle import verify_runtime
    verify_runtime(RUNTIME_ROOT, PINS['runtime']['manifest_sha256'])
else:
    print('所有操作關閉；没有下載、安裝、模型載入、GPU 執行或輸出寫入。')


In [ ]:
if ACTIVE:
    import subprocess
    from tools.arc_next_offline_runtime import launch_phase
    selected = next(f for f in PHASES if globals()[f])
    config = dict(runtime_root=str(RUNTIME_ROOT), model_root=str(MODEL_ROOT), wheels_root=str(WHEELS_ROOT),
        environment_dir=str(ENVIRONMENT), pins=PINS, run_tag=RUN_TAG, output_root=str(BASE),
        training_profile=TRAINING_PROFILE, enable_training_pilot=ENABLE_TRAINING_PILOT,
        max_wall_seconds=MAX_WALL_SECONDS, approved_max_cost_usd=APPROVED_MAX_COST_USD,
        estimated_hourly_cost_usd=ESTIMATED_HOURLY_COST_USD, adapter_path=ADAPTER_PATH)
    result = launch_phase(selected, config, enabled=True)
    print(json.dumps(result, ensure_ascii=False, indent=2))
